# Download a Numerai dataset with `numerapi`

This notebook uses the [`numerapi`](https://github.com/numerai/numerapi) Python client to pull data from the Numerai tournament API — no API key needed for public datasets.

We'll:
1. List the datasets available for the current data version.
2. Download the feature metadata (`features.json`) and a training file.
3. Load the training file with pandas and take a quick look.

In [2]:
from pathlib import Path

from numerapi import NumerAPI

napi = NumerAPI()

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

In [3]:
all_datasets = napi.list_datasets()
v5_datasets = sorted(f for f in all_datasets if f.startswith("v5.0"))
v5_datasets

['v5.0/features.json',
 'v5.0/live.parquet',
 'v5.0/live_benchmark_models.parquet',
 'v5.0/live_example_preds.csv',
 'v5.0/live_example_preds.parquet',
 'v5.0/meta_model.parquet',
 'v5.0/train.parquet',
 'v5.0/train_benchmark_models.parquet',
 'v5.0/validation.parquet',
 'v5.0/validation_benchmark_models.parquet',
 'v5.0/validation_example_preds.csv',
 'v5.0/validation_example_preds.parquet']

## Download the feature metadata

`v5.0/features.json` lists every feature and groups them into feature sets (`small`, `medium`, `all`, ...) plus the available targets. We'll need this to know which columns to load.

In [3]:
import json

napi.download_dataset("v5.0/features.json", str(DATA_DIR / "features.json"))

with open(DATA_DIR / "features.json") as f:
    feature_metadata = json.load(f)

small_features = feature_metadata["feature_sets"]["small"]
print(f"{len(small_features)} features in the 'small' feature set")
small_features[:5]

2026-08-16 20:26:25,367 INFO numerapi.utils: target file already exists
2026-08-16 20:26:25,368 INFO numerapi.utils: download complete


42 features in the 'small' feature set


['feature_antistrophic_striate_conscriptionist',
 'feature_bicameral_showery_wallaba',
 'feature_bridal_fingered_pensioner',
 'feature_collectivist_flaxen_gueux',
 'feature_concurring_fabled_adapter']

## Download a training file

`v5.0/train.parquet` has every era and every feature — it's large. For this lesson we only need a handful of eras to compute feature exposure and try neutralization, so we pass `filters` (pandas `read_parquet` filter syntax) to `download_dataset` and only pull a few eras' worth of rows.

In [4]:
train_sample_path = DATA_DIR / "train_sample.parquet"
if not train_sample_path.exists():
    napi.download_dataset(
        "v5.0/train.parquet",
        str(train_sample_path)
    )

## Load it and take a look

We further narrow to the `small` feature set columns plus `era` and one target, since that's all we'll need for exposure/neutralization work.

In [5]:
import pandas as pd

target = "target"
columns = ["era"] + small_features + [target]

df = pd.read_parquet(DATA_DIR / "train_sample.parquet", columns=columns)
print(df.shape)
df.head()

(2746268, 44)


,era,feature_antistrophic_striate_conscriptionist,feature_bicameral_showery_wallaba,feature_bridal_fingered_pensioner,feature_collectivist_flaxen_gueux,feature_concurring_fabled_adapter,feature_crosscut_whilom_ataxy,feature_departmental_inimitable_sentencer,feature_dialectal_homely_cambodia,feature_donnard_groutier_twinkle,...,feature_trimeter_soggy_greatest,feature_unanalyzable_excusable_whirlwind,feature_unbreakable_constraining_hegelianism,feature_unformed_bent_smatch,feature_unministerial_unextenuated_teleostean,feature_unmodish_zymogenic_rousing,feature_unsystematized_subcardinal_malaysia,feature_willful_sere_chronobiology,feature_zoological_peristomial_scute,target
id,,,,,,,,,,,,,,,,,,,,,
n0007b5abb0c3a25,0001,2,2,2,2,2,0,1,2,2,...,1,1,3,0,2,2,3,3,2,0.25
n003bba8a98662e4,0001,2,2,2,2,2,1,4,2,2,...,2,0,0,0,2,2,4,4,2,0.25
n003bee128c2fcfc,0001,2,2,2,2,2,2,2,2,2,...,1,1,0,1,2,2,0,3,2,0.75
n0048ac83aff7194,0001,2,2,2,2,2,1,4,2,2,...,3,4,1,2,2,2,2,0,2,0.25
n0055a2401ba6480,0001,2,2,2,2,2,0,0,2,2,...,0,1,0,0,2,2,1,4,2,0.25


## Train a LightGBM model on the `small` feature set

Fit a `LGBMRegressor` on `small_features` to predict `target`, using Numerai's typical hyperparameters for a quick baseline model.

In [ ]:
import lightgbm as lgb

model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=5,
    num_leaves=2 ** 5,
    colsample_bytree=0.1,
    verbose=-1,
)

model.fit(df[small_features], df[target])

df["prediction"] = model.predict(df[small_features])
df[["era", target, "prediction"]].head()

In [ ]:
import numpy as np
from scipy.stats import spearmanr

PREDICTION_NAME = "prediction"


def feature_exposures(df):
    feature_names = [f for f in df.columns
                     if f.startswith("feature")]
    exposures = []
    for f in feature_names:
        fe = spearmanr(df[PREDICTION_NAME], df[f])[0]
        exposures.append(fe)
    return np.array(exposures)


def max_feature_exposure(df):
    return np.max(np.abs(feature_exposures(df)))


def feature_exposure(df):
    return np.sqrt(np.mean(np.square(feature_exposures(df))))

In [ ]:
nunique = df[small_features].nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
print(constant_cols)